# Transcribe Danish interviews (Whisper + who-spoke-when)

Runs fully **local** on the GPU. Audio never leaves the machine. For each audio file you get
`<name>-transcription.json` and `<name>-transcription_edit.docx` next to it.

## How to use (3 steps)

1. **Start the right machine on UCloud.** App **JupyterLab**, Flavor **Base**,
   Version **4.6.3**, Machine type **gpu-nvidia-b200** (1 MIG / fractional GPU is enough).
   Mount the folder with the audio files.
   *Version 4.6.3 is the one this was verified on (15 Sep 2026). Newer versions usually
   work too because the code repairs itself, but if something breaks, pick 4.6.3.*
2. **Token (one time):** a Hugging Face token whose account accepted the terms of
   [pyannote/speaker-diarization-3.1](https://hf.co/pyannote/speaker-diarization-3.1) and
   [pyannote/segmentation-3.0](https://hf.co/pyannote/segmentation-3.0). Either it is
   already in the file `.env` in the repo root (`HF_TOKEN=hf_...`), or paste it once into
   `HF_TOKEN` in the next cell: it is then saved to `.env` for you, and you should
   **delete it from the cell again** so it never ends up in git.
3. **Set the folder in the next cell**, then menu **Run → Run All Cells**. Wait. Done.

**How long does it take?** Every line of output has a clock and the time elapsed. The models
(~3 GB) are kept in `/work/speech/models/huggingface`, so they are downloaded only the very
first time (1–5 min); later jobs load them in seconds. Then each interview runs at roughly
10–15× realtime on the GPU: a 30-minute recording takes about 2–3 minutes. A progress bar
shows the transcription of each file, and after the first file the output prints an
estimate for the rest.

Files that already have a transcription are skipped, so you can simply run again after
adding files or after an interruption. If the output says **RESTART THE KERNEL**:
menu **Kernel → Restart Kernel…**, then Run All again.

**If Run All does nothing** (no output after a minute, with or without `[*]`, even after a
kernel restart; normally the first line appears within seconds): this is a bug in UCloud's
JupyterLab that happens when a notebook was left open and idle, and after "Restart Kernel
and Run All". Fix: menu **Kernel → Shut Down Kernel**, then click the kernel name top-right
(it says *No Kernel*) and choose **Python 3**, then Run All again. Reloading the page or
restarting the kernel does **not** fix it; shutting it down and starting a new one does.

Everything else (installing packages, GPU checks, downloading models, a self-test) is
automatic. All code lives in [`transcribe/`](transcribe/) next to this notebook — nothing
to edit here.


In [ ]:
PATH_AUDIO = "/work/speech/Peter"   # folder with the audio files
NUM_SPEAKERS = 2                     # how many people talk in each recording (None = auto-detect)
HF_TOKEN = ""                        # leave empty if the token is already in .env; otherwise paste it here ONCE (it gets saved to .env, then remove it here)


In [ ]:
import pathlib, sys

_here = pathlib.Path.cwd()
_src = next((d for p in [_here, *_here.parents] for d in (p, p / "speech_to_text")
             if (d / "transcribe" / "env.py").is_file()), None)
assert _src, "Open this notebook from inside the 4dpicture-danish repo ('speech_to_text/transcribe' not found)."
sys.path.insert(0, str(_src))

from transcribe import run

run(PATH_AUDIO, num_speakers=NUM_SPEAKERS, hf_token=HF_TOKEN or None)
